<a href="https://colab.research.google.com/github/giulia-belgiovine/Memorability/blob/main/fastAI_mem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from warnings import filterwarnings
filterwarnings('ignore')

import pandas as pd
from fastai.vision.all import *

from plotnine import *
import plotnine.options as plotnine_opts

In [3]:
from fastai.callback.core import Callback
from fastai.metrics import accuracy

def get_loss_curves(recorder):
    loss_curves = []
    for key, loss_curve in recorder.metrics.items():
        subkeys = ['split', 'metric', 'rate']
        loss_type = dict(zip(subkeys, key.split('_')))
        for step, loss in enumerate(loss_curve):
            loss_curves.append({**loss_type, 'step': step+1, 'value': loss})

    return pd.DataFrame(loss_curves)

class ExtendedMetricsRecorder(Callback):
    def before_fit(self):
        self.metrics = {
            'train_loss_batch': [], 'valid_loss_batch': [],
            'train_loss_epoch': [], 'valid_loss_epoch': [],
            'train_accuracy_batch': [], 'valid_accuracy_batch': [],
            'train_accuracy_epoch': [], 'valid_accuracy_epoch': []
        }

    def after_batch(self):
        batch_accuracy = accuracy(self.pred, self.y).item()
        if self.training:
            self.metrics['train_loss_batch'].append(self.loss.item())
            self.metrics['train_accuracy_batch'].append(batch_accuracy)
        else:
            self.metrics['valid_loss_batch'].append(self.loss.item())
            self.metrics['valid_accuracy_batch'].append(batch_accuracy)

    def after_epoch(self):
        # Compute average metrics for the epoch
        for split, n in [('train', self.n_iter), ('valid', len(self.dls.valid))]:
            self.metrics[f'{split}_loss_epoch'].append(
                sum(self.metrics[f'{split}_loss_batch'][-n:]) / n
            )
            self.metrics[f'{split}_accuracy_epoch'].append(
                sum(self.metrics[f'{split}_accuracy_batch'][-n:]) / n
            )

    def to_dataframe(self, which='batch'):
        return pd.DataFrame({key: value for key, value in
                             self.metrics.items() if which in key})

Dataset splits

In [ ]:
memcat_data = pd.read_csv('/content/drive/MyDrive/BMM_2023/projects/Datasets/MemCat_data/memcat_image_data.csv').iloc[:,1:]
#memcat_meta = pd.read_csv('MemCat_data/memcat_raw_memory_data.csv').iloc[:,1:]
path_to_images = '/content/drive/MyDrive/BMM_2023/projects/Datasets/MemCat' # define image set root path

In [ ]:
memcat_data

In [ ]:
np.random.seed(0)
final_list_train = []
final_list_test = []

class_dict = {"animal": 0, "food":1, "landscape":2, "sports":3, "vehicle":4}
for k, v in class_dict.items():

    subframe = memcat_data.loc[memcat_data["category"]==k]

    #Take 200 random samples from test
    df_val = subframe.sample(n=200)
    df_train = subframe.drop(df_val.index)

    # Perform a median split based on mem. score on the remaining train set.
    # Top 50th percentile put in group '2'. Bottom 50th percentile be put in group '1'
    # score = df_train["memorability_w_fa_correction"]
    # df_train["median_split"] = (score < score.quantile()).replace({True:1, False:2})

    # low_score_df = df_train.loc[df_train["median_split"] == 1]
    # high_score_df = df_train.loc[df_train["median_split"] == 2]
    # print("Lenght of first median split: {}. Lenght of second median split: {}.".format(len(low_score_df), len(high_score_df)))

    # Take 800 from low_score and 800 from hig score to remove the ones too close to median
    # composed_dataset = pd.concat([high_score_df[:800], low_score_df[-800:]])
    # composed_dataset = composed_dataset.sort_values(by='memorability_w_fa_correction', ascending=True)

    final_list_train.append(df_train)
    final_list_test.append(df_val)

train_df = pd.concat(final_list_train)
test_df = pd.concat(final_list_test)
print("Total lenght of: Train Dataset --> {}. Test Dataset --> {}".format(len(train_df), len(test_df)))

Total lenght of: Train Dataset --> 9000. Test Dataset --> 1000


In [ ]:
target_cols = ['image_file','category','subcategory']
df = train_df.copy()[target_cols]
df['memory'] = train_df['memorability_w_fa_correction']

# Calculate the median memory for each subcategory
median_memory = df.groupby('subcategory')['memory'].transform('median')

# Create new columns based on the median memory for each subcategory
df['memory_hilow'] = (df['memory'] <= median_memory).astype(int)
df['memory_lowhi'] = (df['memory'] >= median_memory).astype(int)

df['image_file'] = df['category'] + '/' + df['subcategory'] + '/' + df['image_file']

In [ ]:
df

,image_file,category,subcategory,memory,memory_hilow,memory_lowhi
0,animal/bear/000000003481.jpg,animal,bear,0.520408,1,0
1,animal/bear/000000005745.jpg,animal,bear,0.740741,0,1
2,animal/bear/000000011552.jpg,animal,bear,0.673077,1,0
3,animal/bear/000000027439.jpg,animal,bear,0.780702,0,1
4,animal/bear/000000055601.jpg,animal,bear,0.696078,1,0
...,...,...,...,...,...,...
9995,vehicle/yacht/n04610013_9276.jpg,vehicle,yacht,0.509804,1,0
9996,vehicle/yacht/n04610013_9444.jpg,vehicle,yacht,0.685185,0,1
9997,vehicle/yacht/n04610013_9569.jpg,vehicle,yacht,0.546392,1,0
9998,vehicle/yacht/n04610013_9664.jpg,vehicle,yacht,0.552381,1,0


In [ ]:
def run_model_assay(df, path_to_images, sampling='random', labels='subcategory'):
    if sampling == 'random':
        sampling_kwarg = {'valid_pct': 0.5}
    if sampling == 'low_memory':
        sampling_kwarg = {'valid_col': 'memory_lowhi'}
    if sampling == 'high_memory':
        sampling_kwarg = {'valid_col': 'memory_hilow'}

    check = (df.query(f"{sampling_kwarg['valid_col']} == 0")
             ['memory'].agg(['count','mean']))

    print(f"{sampling} sampling: {check['count']} training samples; " +
          f"{round(check['mean'], 3)} average memorability...")

    dls = ImageDataLoaders.from_df(df, path=path_to_images, fn_col='image_file',
                                   label_col=labels, item_tfms=Resize(224), **sampling_kwarg)

    metrics_recorder = ExtendedMetricsRecorder()
    learn = vision_learner(dls, resnet18, metrics=accuracy, cbs=[metrics_recorder])

    # Train the model
    learn.fine_tune(5)
    loss_curves = get_loss_curves(metrics_recorder)
    score = learn.validate()[1]

    return score, loss_curves

In [ ]:
all_results = {sampling: {f'assay_{i}': None for i in [1,2,3,4,5]}
               for sampling in ['low_memory','high_memory']}

In [ ]:
all_results

{'low_memory': {'assay_1': None,
  'assay_2': None,
  'assay_3': None,
  'assay_4': None,
  'assay_5': None},
 'high_memory': {'assay_1': None,
  'assay_2': None,
  'assay_3': None,
  'assay_4': None,
  'assay_5': None}}

In [ ]:
for sampling in ['low_memory','high_memory']:
    for assay in all_results[sampling]:
         all_results[sampling][assay] = run_model_assay(df, path_to_images, sampling)

low_memory sampling: 4958.0 training samples; 0.624 average memorability...


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 256MB/s]


epoch,train_loss,valid_loss,accuracy,time


KeyboardInterrupt: ignored

In [ ]:
scores = {sampling: [result[0] for result in all_results[sampling].values()]
              for sampling in all_results}

In [ ]:
all_results['low_memory']['assay_1'][1]

In [ ]:
(np.mean(scores['low_memory']), np.mean(scores['high_memory']))

In [ ]:
curves = {sampling: pd.concat([result[1] for result in all_results[sampling].values()])
              for sampling in all_results}

In [ ]:
loss_curves = []
for sampling, assays in all_results.items():
    for assay, results in enumerate(assays.values()):
        results = results[1].copy()
        results.insert(0, 'sampling', sampling)
        results.insert(1, 'assay', assay + 1)
        loss_curves.append(results)

loss_curves = pd.concat(loss_curves)

In [ ]:
target_rate, target_metric = 'epoch', 'accuracy'

plotnine_opts.figure_size = (10,5)

plot_data = (loss_curves.copy().query('rate == @target_rate')
             .query('metric == @target_metric'))

plot_data['assay_sampling'] = (plot_data['sampling'] + plot_data['split'] +
                                   plot_data['assay'].astype(str))

mapping = {'x': 'step', 'y': 'value', 'color': 'sampling',
           'linetype': 'split', 'group':'assay_sampling'}

(ggplot(plot_data, aes(**mapping)) + geom_line() + ylim([0,1]) + theme_bw()).draw()